# 面试题：海量流中怎样用 Count-Min Sketch 与 Space-Saving 找 Top-K？

精确 Counter 的内存随 distinct key 增长。Count-Min Sketch 用固定二维计数器提供只高估的频次；Space-Saving 用固定槽位维护 heavy hitters。本 Notebook 手写稳定 hash、误差界、保守更新、合并、Top-K、窗口轮换、溢出和快照。

不调用流处理/Sketch 库；受控 Zipf 流用于比较精确计数、误差和候选召回。

In [ ]:
import copy,hashlib,heapq,json,math,warnings  # 导入本单元所需的依赖。
from collections import Counter  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
RNG75=np.random.default_rng(7501)  # 计算并保存当前步骤的中间状态。
def canonical75(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha75(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert RNG75 is not None  # 用受控断言验证关键不变量。

## 1. 稳定多行 hash

每一行需要不同 seed；使用 BLAKE2b 的 keyed/personalization bytes，避免 Python hash 跨进程随机。key 统一编码 UTF-8，行号进入 hash，结果模 width。

hash family、width/depth、counter dtype 和编码都是可合并协议；任一不同都不能直接逐元素相加。

In [ ]:
def bucket75(key,row,width,seed=7501):  # 定义本节可复用的核心函数。
    if not isinstance(key,str) or not 0<=row<256 or width<1: raise ValueError("hash_contract")  # 按当前条件选择后续控制路径。
    h=hashlib.blake2b(key.encode(),digest_size=8,person=(seed+row).to_bytes(8,"little")); return int.from_bytes(h.digest(),"little")%width  # 计算并保存当前步骤的中间状态。
assert bucket75("search",0,100)==bucket75("search",0,100)  # 用受控断言验证关键不变量。
assert 0<=bucket75("搜索",2,17)<17 and any(bucket75("search",r,100)!=bucket75("search",0,100) for r in range(1,4))  # 用受控断言验证关键不变量。
try: bucket75(None,0,10); raise AssertionError("non-string key accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="hash_contract"  # 捕获预期异常并验证失败分支。

## 2. Count-Min Sketch 与保守更新

标准 update 将 key 对应的每行 counter 都加 count；query 取最小值，因此碰撞只会高估。conservative update 只把低于目标估计的 counter 提到目标，可减少不必要高估，但分布式 merge 语义不再等同于标准线性 sketch。

理论上 width≈`e/ε`、depth≈`ln(1/δ)`，以概率 `1-δ` 保证误差不超过 `ε * total_count`。

In [ ]:
class CountMinSketch75:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,width,depth,seed=7501,dtype=np.uint64,conservative=False):  # 定义本节可复用的核心函数。
        if width<1 or depth<1: raise ValueError("cms_config")  # 按当前条件选择后续控制路径。
        self.width=width; self.depth=depth; self.seed=seed; self.table=np.zeros((depth,width),dtype=dtype); self.total=0; self.conservative=conservative  # 计算并保存当前步骤的中间状态。
    def estimate(self,key): return int(min(self.table[r,bucket75(key,r,self.width,self.seed)] for r in range(self.depth)))  # 定义本节可复用的核心函数。
    def update(self,key,count=1):  # 定义本节可复用的核心函数。
        if not isinstance(count,int) or count<0: raise ValueError("count_contract")  # 按当前条件选择后续控制路径。
        if count==0:return self.estimate(key)  # 按当前条件选择后续控制路径。
        positions=[bucket75(key,r,self.width,self.seed) for r in range(self.depth)]  # 计算并保存当前步骤的中间状态。
        if self.conservative:  # 按当前条件选择后续控制路径。
            target=self.estimate(key)+count  # 计算并保存当前步骤的中间状态。
            for r,p in enumerate(positions): self.table[r,p]=max(self.table[r,p],target)  # 遍历输入元素以累积或检查结果。
        else:  # 处理前置条件不成立的分支。
            for r,p in enumerate(positions): self.table[r,p]+=count  # 遍历输入元素以累积或检查结果。
        self.total+=count; return self.estimate(key)  # 计算并保存当前步骤的中间状态。
    def merge(self,other):  # 定义本节可复用的核心函数。
        if (self.width,self.depth,self.seed,self.conservative)!=(other.width,other.depth,other.seed,other.conservative) or self.conservative: raise ValueError("merge_protocol")  # 按当前条件选择后续控制路径。
        out=CountMinSketch75(self.width,self.depth,self.seed,self.table.dtype); out.table=self.table+other.table; out.total=self.total+other.total; return out  # 计算并保存当前步骤的中间状态。
cms_probe75=CountMinSketch75(16,3); cms_probe75.update("a",3); cms_probe75.update("b",2)  # 计算并保存当前步骤的中间状态。
assert cms_probe75.estimate("a")>=3 and cms_probe75.estimate("b")>=2 and cms_probe75.total==5  # 用受控断言验证关键不变量。
assert cms_probe75.estimate("missing")>=0 and cms_probe75.table.shape==(3,16)  # 用受控断言验证关键不变量。
try: cms_probe75.update("a",-1); raise AssertionError("negative count accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="count_contract"  # 捕获预期异常并验证失败分支。

## 3. Zipf 流与误差 oracle

构造 300 个 key、30,000 次事件的 Zipf 流。精确 Counter 仅用于评估。CMS width=512/depth=5 固定内存，与 distinct 数无关。检查没有低估，并将最大误差与 `εN` 理论量级比较。

真流量会有突发、攻击 key 和时间漂移，应按切片评估而非只看 Zipf。

In [ ]:
keys75=[f"term-{i:03d}" for i in range(300)]; probs75=1/(np.arange(1,301)**1.12); probs75=probs75/probs75.sum(); stream75=RNG75.choice(keys75,size=30000,p=probs75).tolist()  # 计算并保存当前步骤的中间状态。
exact75=Counter(stream75); cms75=CountMinSketch75(512,5); conservative75=CountMinSketch75(512,5,conservative=True)  # 计算并保存当前步骤的中间状态。
for key in stream75: cms75.update(key); conservative75.update(key)  # 遍历输入元素以累积或检查结果。
errors75={k:cms75.estimate(k)-v for k,v in exact75.items()}; conservative_errors75={k:conservative75.estimate(k)-v for k,v in exact75.items()}  # 计算并保存当前步骤的中间状态。
assert all(e>=0 for e in errors75.values()) and all(e>=0 for e in conservative_errors75.values())  # 用受控断言验证关键不变量。
assert max(errors75.values())<=math.ceil(math.e/512*len(stream75))*2  # 用受控断言验证关键不变量。
assert np.mean(list(conservative_errors75.values()))<=np.mean(list(errors75.values()))  # 用受控断言验证关键不变量。
assert cms75.total==len(stream75) and len(exact75)==300  # 用受控断言验证关键不变量。

## 4. Space-Saving 固定槽位 Top-K

若 key 已在表中就加一；有空槽直接插入；满时替换最小 counter 的 key，新 counter=`min+count`，并记录 error=`min`。真实频次落在 `[estimate-error, estimate]`。

教学实现每次线性找最小槽，复杂度 `O(capacity)`；生产用 heap+lazy version 或专用结构降到对数。

In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class Slot75: count:int; error:int  # 定义承载本节状态与行为的数据结构。
class SpaceSaving75:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,capacity):  # 定义本节可复用的核心函数。
        if capacity<1: raise ValueError("capacity_contract")  # 按当前条件选择后续控制路径。
        self.capacity=capacity; self.slots={}  # 计算并保存当前步骤的中间状态。
    def update(self,key,count=1):  # 定义本节可复用的核心函数。
        if count<1: raise ValueError("positive_count_required")  # 按当前条件选择后续控制路径。
        if key in self.slots: self.slots[key].count+=count  # 按当前条件选择后续控制路径。
        elif len(self.slots)<self.capacity: self.slots[key]=Slot75(count,0)  # 按当前条件选择后续控制路径。
        else:  # 处理前置条件不成立的分支。
            victim=min(self.slots,key=lambda k:(self.slots[k].count,k)); base=self.slots.pop(victim).count; self.slots[key]=Slot75(base+count,base)  # 计算并保存当前步骤的中间状态。
    def topk(self,k): return sorted(((key,s.count,s.error) for key,s in self.slots.items()),key=lambda x:(-x[1],x[0]))[:k]  # 定义本节可复用的核心函数。
ss_probe75=SpaceSaving75(2)  # 计算并保存当前步骤的中间状态。
for x in ["a","b","a","c"]: ss_probe75.update(x)  # 遍历输入元素以累积或检查结果。
assert len(ss_probe75.slots)==2 and ss_probe75.topk(1)[0][1]>=2  # 用受控断言验证关键不变量。
assert all(count-error>=0 for _,count,error in ss_probe75.topk(2))  # 用受控断言验证关键不变量。
try: SpaceSaving75(0); raise AssertionError("zero capacity accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="capacity_contract"  # 捕获预期异常并验证失败分支。

## 5. Heavy-hitter 召回与误差区间

用 capacity=40 扫流，比较 exact Top-10。Space-Saving 的候选再用 CMS 估计排序；若需要精确榜单，可对候选回查持久日志。Top-K 指标关注 set recall、rank correlation、最低 heavy-hitter gap 和内存。

当第 K/K+1 频次很接近时，任何近似结构都可能不稳定，应返回置信区间或扩大候选。

In [ ]:
ss75=SpaceSaving75(40)  # 计算并保存当前步骤的中间状态。
for key in stream75: ss75.update(key)  # 遍历输入元素以累积或检查结果。
exact_top75=[k for k,_ in exact75.most_common(10)]; candidate_top75=[k for k,_,_ in ss75.topk(20)]; recall_top75=len(set(exact_top75)&set(candidate_top75))/10  # 计算并保存当前步骤的中间状态。
reranked75=sorted(candidate_top75,key=lambda k:(-cms75.estimate(k),k))[:10]  # 计算并保存当前步骤的中间状态。
rerank_recall75=len(set(exact_top75)&set(reranked75))/10  # 计算并保存当前步骤的中间状态。
assert recall_top75==1. and rerank_recall75==1.  # 用受控断言验证关键不变量。
assert len(ss75.slots)==40 and len(candidate_top75)==20  # 用受控断言验证关键不变量。
for key,count,error in ss75.topk(20): assert count-error<=exact75[key]<=count  # 遍历输入元素以累积或检查结果。
assert exact75[exact_top75[0]]>exact75[exact_top75[-1]]  # 用受控断言验证关键不变量。

## 6. 分片线性合并与协议拒绝

标准 CMS 可逐元素相加：分别处理偶数/奇数分片后 merge，结果应与单流 sketch 完全相同。conservative update 不是线性结构，本例明确拒绝 merge，避免给出看似合理但语义错误的结果。

Space-Saving 合并需要误差传播与再压缩，这里不伪装成简单字典相加；生产可让分片上报候选、中心回查 CMS/精确计数。

In [ ]:
left75=CountMinSketch75(512,5); right75=CountMinSketch75(512,5)  # 计算并保存当前步骤的中间状态。
for i,key in enumerate(stream75): (left75 if i%2==0 else right75).update(key)  # 遍历输入元素以累积或检查结果。
merged75=left75.merge(right75)  # 计算并保存当前步骤的中间状态。
assert np.array_equal(merged75.table,cms75.table) and merged75.total==cms75.total  # 用受控断言验证关键不变量。
assert all(merged75.estimate(k)==cms75.estimate(k) for k in exact_top75)  # 用受控断言验证关键不变量。
try: left75.merge(CountMinSketch75(256,5)); raise AssertionError("incompatible sketches merged")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="merge_protocol"  # 捕获预期异常并验证失败分支。
try: conservative75.merge(conservative75); raise AssertionError("conservative sketch merged")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="merge_protocol"  # 捕获预期异常并验证失败分支。

## 7. 时间窗口、轮换与溢出

CMS 不支持任意删除。滑窗常使用分钟/小时 sketch ring，查询时合并最近桶；过期直接丢整个桶。衰减计数会破坏整数与理论界，必须另定义。

counter dtype 会溢出；这里用 uint64 并在 update 前检查上限的生产要求。教学流很小，但 manifest 仍绑定 dtype。

In [ ]:
buckets_window75=[]  # 计算并保存当前步骤的中间状态。
for part in np.array_split(np.array(stream75,dtype=object),3):  # 遍历输入元素以累积或检查结果。
    sketch=CountMinSketch75(512,5)  # 计算并保存当前步骤的中间状态。
    for key in part.tolist(): sketch.update(key)  # 遍历输入元素以累积或检查结果。
    buckets_window75.append(sketch)  # 执行当前语句以推进本节示例。
last_two75=buckets_window75[1].merge(buckets_window75[2]); exact_last_two75=Counter(stream75[len(stream75)//3:])  # 计算并保存当前步骤的中间状态。
assert all(last_two75.estimate(k)>=v for k,v in exact_last_two75.items())  # 用受控断言验证关键不变量。
assert last_two75.total==len(stream75)-len(stream75)//3  # 用受控断言验证关键不变量。
assert np.iinfo(cms75.table.dtype).max>cms75.table.max()  # 用受控断言验证关键不变量。

## 8. 发布与面试总结

manifest 绑定 width/depth/seed/hash/dtype/update mode、窗口边界、table bytes 和 Space-Saving capacity。loader 从实际 array 重算摘要；不同协议 sketch 必须 fail closed。

面试回答顺序：精确计数为什么放不下 → CMS 误差界 → Space-Saving 候选 → Top-K 精排 → 合并/窗口 → 溢出和监控。明确 CMS 不直接枚举 key，所以需要候选结构。

In [ ]:
def table_digest75(table):  # 定义本节可复用的核心函数。
    a=np.ascontiguousarray(table); return sha75(str(a.dtype).encode()+canonical75(list(a.shape)).encode()+a.tobytes())  # 计算并保存当前步骤的中间状态。
manifest75={"artifact_id":"stream-topk-v1","cms":{"width":512,"depth":5,"seed":7501,"hash":"blake2b-personalized-v1","dtype":str(cms75.table.dtype),"update":"standard"},"space_saving":{"capacity":40},"table_digest":table_digest75(cms75.table),"total":cms75.total,"window":"all-controlled-stream"}  # 计算并保存当前步骤的中间状态。
TRUST75=MappingProxyType({manifest75["artifact_id"]:sha75(canonical75(manifest75).encode())})  # 计算并保存当前步骤的中间状态。
def load_sketch75(m,sketch):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["table_digest"]=table_digest75(sketch.table); actual["total"]=sketch.total  # 计算并保存当前步骤的中间状态。
    if TRUST75.get(actual.get("artifact_id"))!=sha75(canonical75(actual).encode()): raise RuntimeError("untrusted_stream_sketch")  # 按当前条件选择后续控制路径。
    return sketch  # 返回当前分支计算出的结果。
loaded75=load_sketch75(manifest75,cms75)  # 计算并保存当前步骤的中间状态。
assert loaded75.estimate(exact_top75[0])>=exact75[exact_top75[0]] and isinstance(TRUST75,MappingProxyType)  # 用受控断言验证关键不变量。
forged_sketch75=copy.deepcopy(cms75); forged_sketch75.table[0,0]+=1  # 计算并保存当前步骤的中间状态。
try: load_sketch75(manifest75,forged_sketch75); raise AssertionError("forged sketch accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_stream_sketch"  # 捕获预期异常并验证失败分支。
print({"distinct":len(exact75),"top10_recall":rerank_recall75,"max_error":max(errors75.values()),"memory_counters":cms75.table.size})  # 执行当前语句以推进本节示例。

## 9. 复杂度、失败模式与来源

CMS 每次 update/query 是 `O(depth)`，内存 `width*depth`；Space-Saving 教学版 update `O(capacity)`。常见错误：用不稳定 hash、把 estimate 当无偏值、CMS 单独找不到 key、合并不同 seed、conservative sketch 线性合并、滑窗逐 key 删除和 uint counter 溢出。

- Cormode & Muthukrishnan, [An Improved Data Stream Summary: The Count-Min Sketch](https://dimacs.rutgers.edu/~graham/pubs/papers/cm-full.pdf)。
- Metwally et al., [Efficient Computation of Frequent and Top-k Elements in Data Streams](https://www.cse.ust.hk/~raywong/comp5331/References/frequent.pdf)，Space-Saving。
- Manku & Motwani, [Approximate Frequency Counts over Data Streams](https://www.vldb.org/conf/2002/S10P03.pdf)。